In [1]:
top_features = [
    'session_recency_min',
    'purchase_number_stddev',
    'purchase_number_cv',
    'purchase_recency_min',
    'purchase_count_month_lag0',
    'purchase_count_month_ma3',
    'purchase_revenue_month_lag0',
    'session_count_month_lag0'
]

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import pickle

df = pd.read_csv(
    'data/processed/selected_features.csv'
)

X = df[top_features]
y = df['target_event']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='logloss'
)

model.fit( X_train, y_train) 

pred_prob = model.predict_proba(X_test)[:,1]

print("ROC-AUC:",roc_auc_score(y_test,pred_prob))

ROC-AUC: 0.9011858536974803


In [3]:
with open('models/churn_model_simple.pkl','wb') as file:
    pickle.dump(model,file)

---
The original XGBoost model was trained using 30 selected features and achieved a ROC-AUC score of 0.9075.

To improve usability in the Streamlit application, a simplified deployment model was created using the top 8 most important business features.

The simplified model achieved a ROC-AUC score of 0.9012, demonstrating only a minimal reduction in predictive performance.

This result indicates that customer churn can be effectively predicted using a small set of highly informative behavioral features.

The simplified model was selected for deployment because it offers:

• Easier user interaction
• Faster predictions
• Better interpretability
• Near-identical predictive performance

The model was saved for integration into the Streamlit application.

In [4]:
print(model.classes_)

[0 1]


In [5]:
df['target_event'].value_counts()

target_event
0    76480
1    36130
Name: count, dtype: int64

In [6]:
df.groupby('target_event')[
    [
        'session_recency_min',
        'purchase_recency_min',
        'purchase_count_month_lag0'
    ]
].mean()

,session_recency_min,purchase_recency_min,purchase_count_month_lag0
target_event,,,
0,4.671215,8.872219,2.080086
1,31.650634,37.040739,0.422087


In [7]:
df[
    [
        'purchase_number_stddev',
        'purchase_number_cv'
    ]
].describe()

,purchase_number_stddev,purchase_number_cv
count,112610.000000,112610.000000
mean,6.674153,0.555162
std,6.941477,0.024044
min,0.000000,0.471405
25%,2.738613,0.552771
50%,5.916080,0.564076
75%,8.225975,0.568535
max,503.016400,0.604971


In [8]:
top_features

['session_recency_min',
 'purchase_number_stddev',
 'purchase_number_cv',
 'purchase_recency_min',
 'purchase_count_month_lag0',
 'purchase_count_month_ma3',
 'purchase_revenue_month_lag0',
 'session_count_month_lag0']

In [9]:
test_customer = pd.DataFrame({
    'session_recency_min':[4],
    'purchase_number_stddev':[1.0],
    'purchase_number_cv':[0.5],
    'purchase_recency_min':[7],
    'purchase_count_month_lag0':[11],
    'purchase_count_month_ma3':[12],
    'purchase_revenue_month_lag0':[188],
    'session_count_month_lag0':[6]
})

print(model.predict_proba(test_customer))
print(model.predict(test_customer))

[[0.5656965  0.43430355]]
[0]


In [10]:
bad_customer = pd.DataFrame({
    'session_recency_min':[340],
    'purchase_number_stddev':[1.0],
    'purchase_number_cv':[0.5],
    'purchase_recency_min':[340],
    'purchase_count_month_lag0':[0],
    'purchase_count_month_ma3':[0],
    'purchase_revenue_month_lag0':[0],
    'session_count_month_lag0':[0]
})

print(model.predict_proba(bad_customer))
print(model.predict(bad_customer))

[[0.8568845  0.14311554]]
[0]


In [11]:
from sklearn.metrics import roc_auc_score

pred_prob = model.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test, pred_prob))

ROC AUC: 0.9011858536974803
